In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from tqdm import tqdm
from Bio import SeqIO

# Pseudogene and GC of samples
for genus_name in keep_genus:
    os.chdir(f'{base_folder}/kmer_chr_frag_samples/{genus_name}/fragment_records')
    filenames = os.listdir()
    seq_data = []
    with tqdm(total = len(filenames), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for file in filenames:
            temp_dict = {'sequence': file.replace('.txt', ''), 'CDSs (total)': 0, 'CDSs (with protein)': 0, 'CDSs (without protein, Pseudo Genes)': 0}
            with open(file, 'r') as f:
                raw = f.read()
            kmer_dict = ast.literal_eval(raw)
            temp_dict['GC_content'] = (kmer_dict['1-mer']['G'] + kmer_dict['1-mer']['C'])/sum(kmer_dict['1-mer'].values())
            acc_n, contig, start_site, end_site = file.replace('.txt', '').split('-')
            handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
            seq_record = SeqIO.parse(handle, 'genbank')
            for record in seq_record:
                if record.id != contig:
                    continue
                temp_record = record[int(start_site):int(end_site)]
                for feature in temp_record.features:
                    if feature.type == 'CDS':
                        temp_dict['CDSs (total)'] += 1
                        if 'translation' in feature.qualifiers:
                            temp_dict['CDSs (with protein)'] += 1
                        else:
                            temp_dict['CDSs (without protein, Pseudo Genes)'] += 1
            seq_data.append(pd.DataFrame([temp_dict]))
            pbar.update(1)
    seq_table = pd.concat(seq_data, ignore_index=True)
    os.chdir(f'{base_folder}/kmer_chr_frag_samples/{genus_name}')
    seq_table.to_csv('chromosome_fragment_data.tsv', sep='\t', index=False)

Escherichia: 100%|██████████████████████████████████████████████| 1.00k/1.00k [08:00<00:00, 2.08B/s]
Klebsiella: 100%|███████████████████████████████████████████████| 1.00k/1.00k [08:39<00:00, 1.93B/s]
Staphylococcus: 100%|███████████████████████████████████████████| 1.00k/1.00k [04:12<00:00, 3.97B/s]
Pseudomonas: 100%|██████████████████████████████████████████████| 1.00k/1.00k [09:50<00:00, 1.69B/s]
Bacillus: 100%|█████████████████████████████████████████████████| 1.00k/1.00k [07:10<00:00, 2.32B/s]
Salmonella: 100%|███████████████████████████████████████████████| 1.00k/1.00k [07:40<00:00, 2.17B/s]
Streptococcus: 100%|████████████████████████████████████████████| 1.00k/1.00k [03:15<00:00, 5.12B/s]
Streptomyces: 100%|█████████████████████████████████████████████| 1.00k/1.00k [13:17<00:00, 1.25B/s]
Acinetobacter: 100%|████████████████████████████████████████████| 1.00k/1.00k [05:52<00:00, 2.83B/s]
Enterococcus: 100%|█████████████████████████████████████████████| 1.00k/1.00k [04:32<00:00,